In [1]:
import os
import torch

from BudaOCR.Data import VitConfig
from BudaOCR.Config import CHARSET
from BudaOCR.Networks import Easter2ViTNetwork
from BudaOCR.Encoder import StackEncoder, WylieEncoder
from BudaOCR.Trainer import OCRTrainer

from BudaOCR.Utils import (
    accumulate_distributions,
    build_data_paths,
    create_dir,
    read_stack_file,
    shuffle_data
    )

print(torch.__version__)
torch.cuda.empty_cache()

print(torch.cuda.is_available())

/Users/eric/Desktop/Projects/Python/tibetan-ocr-training/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.10.0
False


In [2]:
# encoders
wylie_encoder = WylieEncoder(CHARSET)

stack_file = "tib-stacks_v2.txt"
stacks = read_stack_file(stack_file)
stack_encoder = StackEncoder(stacks)

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


building ctcvocab: 84
building ctcvocab: 10376


In [3]:
#encoder = wylie_encoder
encoder = wylie_encoder
num_classes = encoder.num_classes
vit_cfg = VitConfig()

print(num_classes)

# train params
image_width = 3200
image_height = 100
batch_size = 32
workers = 4
network = Easter2ViTNetwork(vit_cfg, image_width, image_height, num_classes=num_classes)

84
Using MPS compute backend


#### Single Dataset Training

In [ ]:
# local dir
dataset_path = "D:/Datasets/KhyentseWangpo"
image_paths, label_paths = build_data_paths(dataset_path, img_file_ext="jpg")
image_paths, label_paths = shuffle_data(image_paths, label_paths)

print(f"Images: {len(image_paths)}, Labels: {len(label_paths)}")

output_dir = os.path.join("Output")
create_dir(output_dir)

In [ ]:
ocr_trainer = OCRTrainer(
    network=network,
    label_encoder=encoder,
    workers=workers, 
    image_width=image_width,
    image_height=image_height,
    batch_size=batch_size, 
    output_dir=output_dir, 
    preload_labels=True
    )
ocr_trainer.init(image_paths, label_paths)

In [ ]:
num_epochs = 2
ocr_trainer.train(epochs=num_epochs, patience=10, check_cer=True, export_onnx=True, silent=False)

In [8]:
checkpoint = torch.load("Output/2026_2_4_17_6/OCRModel.pth")
network.model.load_state_dict(checkpoint["state_dict"])

print("Check")

Check


#### Training multiple distributions

In [ ]:
data_root = "../home"
distributions = ["DergeTenjur", "LhasaKanjur", "Karmapa8", "LithangKanjur"]
distribution = accumulate_distributions(data_root, distributions)

In [ ]:
output_dir = "Output"
create_dir(output_dir)


ocr_trainer = OCRTrainer(
    network=network,
    label_encoder=wylie_encoder,
    workers=workers, 
    image_width=image_width,
    image_height=image_height,
    batch_size=batch_size, 
    output_dir=output_dir, 
    preload_labels=True
    )

assert (distribution is not None)
ocr_trainer.init_from_distribution(distribution)

In [ ]:
num_epochs = 12
ocr_trainer.train(epochs=num_epochs, check_cer=True, export_onnx=True, silent=False)